# edge-grader: teacher reasons for stage 3b on Kaggle

Qwen3-14B writes a short reason for every SciEntsBank train answer with `edge-grader-reasons`: it is given the question, the reference answer, the student answer and the **gold 3-way label**, and explains that label in 1–3 sentences without naming one. The reasons are what the stage-3b adapters train on. Design: `docs/superpowers/specs/2026-11-01-reasoning-finetune-design.md` §5.1.

**The teacher is Qwen3-14B** (Qwen's official Q4_K_M GGUF, Apache 2.0), the news judge's model, served in `llama-server` on the same pinned llama.cpp build and flags as the Gemma runs, thinking off, temperature 0, at most 128 tokens.

**`N = 50` is the pilot:** read the reasons and the drop share before the full run (`N = 0`, all 4,969). More than 5% dropped means the prompt is fixed first.

**Before running** (notebook sidebar): GPU T4 x1, Internet on. Code: the private dataset `edge-grader-code` (a `git bundle`, for `kaggle kernels push`), or the `GITHUB_TOKEN` secret as in `kaggle_baseline.ipynb`.

**Output:** `results-reasons.zip` with `results/finetune/reasons-<teacher>[-n<N>]-<stamp>.json` (settings, drop counts, sha256) and `results/finetune/raw/reasons-<teacher>[-n<N>].jsonl` (one reason per answer). Each reason is saved as it comes; if the run stops, the same command resumes.

In [ ]:
N = 50   # the pilot; 0 = the whole train split (4,969 answers, several hours)
TEACHER = ("Qwen/Qwen3-14B-GGUF", "Qwen3-14B-Q4_K_M.gguf")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and install it with uv

In [ ]:
import base64
import os
import shutil
import subprocess
from pathlib import Path

REPO = "/kaggle/working/edge-grader"
# The venv lives outside /kaggle/working: everything there is Output, and downloading results pages
# through every file (a venv in there got the CLI rate-limited, 2026-10-28).
os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/edge-grader-venv"
shutil.rmtree(REPO, ignore_errors=True)
# Runs started with `kaggle kernels push` can't read secrets (HTTP 400, 2026-10-28): those get the
# code from a git bundle in the private dataset edge-grader-code instead of GitHub.
# Only from that dataset, under either mount layout (/kaggle/input/<name>/ or .../datasets/<owner>/<name>/).
bundle = next(Path("/kaggle/input").glob("**/edge-grader-code/edge-grader.bundle"), None)
if bundle:
    clone = subprocess.run(["git", "clone", "-q", str(bundle), REPO], capture_output=True, text=True, check=False)
else:
    from kaggle_secrets import UserSecretsClient

    # The token reaches git through the environment as an HTTP header, as actions/checkout does: never
    # in the command line (a failed clone's error, shown in the notebook output, would print it) and
    # never in .git/config.
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {**os.environ, "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
           "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}
    del token, auth
    clone = subprocess.run(["git", "clone", "-q", "https://github.com/gnotnek/edge-grader.git", REPO],
                           env=env, capture_output=True, text=True, check=False)
    del env
if clone.returncode != 0:
    raise RuntimeError(f"git clone failed (check the GITHUB_TOKEN secret or the code dataset): {clone.stderr.strip()}")
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)

In [ ]:
!pip install -q uv
!cd /kaggle/working/edge-grader && uv sync -q && uv run pytest -q 2>&1 | tail -1

## 2. Build llama-server with CUDA (about 10–15 min, once per session)

The same pinned llama.cpp commit (`a3a1c47`) as every Kaggle run.

In [ ]:
%%bash
set -e
nvidia-smi -L || { echo "No GPU in this session: Settings -> Accelerator -> GPU, then restart the session"; exit 1; }
cd /tmp
# Pinned, so every run uses the same llama.cpp. Unpinned, Kaggle got whatever was newest that day:
# the stage-1 baseline ran on 86ea01d, the RAG-en runs on a3a1c47 (docs/results/rag.md).
LLAMA_CPP_COMMIT=a3a1c4747fdc0dcad40b3946108b89375d9a7d0e
if [ "$(git -C llama.cpp rev-parse HEAD 2>/dev/null)" != "$LLAMA_CPP_COMMIT" ]; then
  rm -rf llama.cpp && mkdir llama.cpp && cd llama.cpp
  git init -q && git remote add origin https://github.com/ggml-org/llama.cpp
  git fetch -q --depth 1 origin "$LLAMA_CPP_COMMIT" && git checkout -q FETCH_HEAD
  cd /tmp
fi
export PATH=/usr/local/cuda/bin:$PATH
CUDA_ROOT=$(dirname "$(dirname "$(readlink -f "$(command -v nvcc)")")")
# CMake can miss libcuda on Kaggle ("links to CUDA::cuda_driver but the target was not found"):
# the driver library sits outside the toolkit (e.g. /usr/local/nvidia/lib64). Find it and pass it in.
LIBCUDA=$( { ls "$CUDA_ROOT"/lib64/stubs/libcuda.so "$CUDA_ROOT"/targets/x86_64-linux/lib/stubs/libcuda.so 2>/dev/null;
             ldconfig -p | awk '/libcuda\.so/ {print $NF}';
             find /usr /opt -name 'libcuda.so*' 2>/dev/null; } | head -1 )
echo "nvcc: $(command -v nvcc)  CUDA_ROOT: $CUDA_ROOT  libcuda: ${LIBCUDA:-NOT FOUND}"
[ -n "$LIBCUDA" ] || { echo "GPU is on but libcuda was not found; paste this output to debug"; exit 1; }
rm -rf llama.cpp/build   # a failed configure caches the bad result
# 75 = T4, 60 = P100. Building only these keeps compile time down.
cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES="60;75" -DLLAMA_CURL=OFF \
  -DCUDAToolkit_ROOT="$CUDA_ROOT" -DCMAKE_CUDA_COMPILER="$CUDA_ROOT/bin/nvcc" \
  -DCUDA_cuda_driver_LIBRARY="$LIBCUDA" > /tmp/cmake.log 2>&1 || { tail -30 /tmp/cmake.log; exit 1; }
cmake --build llama.cpp/build -j"$(nproc)" --target llama-server > /tmp/build.log 2>&1 || { tail -30 /tmp/build.log; exit 1; }
cd llama.cpp && echo "llama.cpp $(git rev-parse --short HEAD)" && ls -la build/bin/llama-server

## 3. Download the teacher

In [ ]:
from huggingface_hub import hf_hub_download

TEACHER_PATH = hf_hub_download(*TEACHER, local_dir="/tmp/models")
TEACHER_PATH

## 4. Write the reasons

Starts `llama-server` with the teacher (the Gemma runs' flags; `-rea off` turns Qwen3's thinking off too), then runs `edge-grader-reasons`.

In [ ]:
import os
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path


def start_server(model_path):
    cmd = ["/tmp/llama.cpp/build/bin/llama-server", "-m", model_path, "-c", "8192",
           "--host", "127.0.0.1", "--port", "8080", "-ngl", "99",
           "-rea", "off", "--reasoning-budget", "-1", "--cache-ram", "0"]
    with open("/tmp/server-teacher.log", "w") as log:
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(150):
        try:
            if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2).status == 200:
                return proc
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            pass  # still loading
        if proc.poll() is not None:
            break
        time.sleep(2)
    proc.kill()
    # /tmp is gone after a Save & Run All version: the cause has to be in the output.
    print("\n".join(Path("/tmp/server-teacher.log").read_text().splitlines()[-40:]))
    raise RuntimeError("llama-server failed; see /tmp/server-teacher.log (last lines above)")


server = start_server(TEACHER_PATH)
try:
    !cd /kaggle/working/edge-grader && uv run edge-grader-reasons --base-url http://127.0.0.1:8080/v1 --n {N}
finally:
    server.terminate()
    server.wait()

## 5. Look at them

The drop share, and the first reasons with their answers. **Gate:** more than 5% dropped → fix the prompt before the full run.

In [ ]:
import json

REPO = Path("/kaggle/working/edge-grader")
[summary] = sorted(REPO.glob("results/finetune/reasons-*.json"))[-1:]
s = json.loads(summary.read_text())
print(summary.name, {k: s[k] for k in ("n", "kept", "drops", "drop_share", "wall_s")})
for line in Path(REPO, s["reasons"]).read_text().splitlines()[:10]:
    r = json.loads(line)
    print(f"[{r['gold']}{' DROP ' + r['drop'] if r['drop'] else ''}] {r['reason']}")

## 6. Download the results
On your machine, from the repo root:
```bash
unzip -n ~/Downloads/results-reasons.zip -d .
```

In [ ]:
import zipfile

from IPython.display import FileLink

out = Path("/kaggle/working/results-reasons.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(REPO.glob("results/finetune/**/reasons-*")):
        z.write(f, f.relative_to(REPO))
print(*zipfile.ZipFile(out).namelist(), sep="\n")
FileLink("results-reasons.zip")